# Pass@1 Benchmark: Qwen2.5-Coder-7B vs Qwen2.5-Coder-7B-Instruct

**Kaggle Notebook version** (adapted from the original Colab notebook).

Evaluates **pass@1** on **HumanEval** and **HumanEval+** (EvalPlus), each under **zero-shot** and **few-shot** prompting, for:
- `Qwen/Qwen2.5-Coder-7B` (base, completion-style prompting)
- `Qwen/Qwen2.5-Coder-7B-Instruct` (chat-template prompting)

That's 2 models x 2 datasets x 2 prompt strategies = 8 runs. Each run's raw generations + pass/fail are saved to `/kaggle/working/data/results_<model>_<dataset>_<template>.json`, and a final summary table is printed/saved at the end.

**Runtime:** Kaggle GPU (P100 or T4 x2 — enable under *Settings → Accelerator*). Models are loaded in 4-bit to fit comfortably in ~16GB VRAM, loaded/unloaded one at a time.

**Before running:**
1. In the notebook editor, go to **Settings → Accelerator** and pick a GPU (T4 x2 or P100).
2. Go to **Settings → Internet** and turn it **On** (needed to download models/datasets from the Hugging Face Hub).
3. Add your Hugging Face token as a Kaggle Secret: **Add-ons → Secrets → Add a new secret**, label it `HF_TOKEN`, and attach it to this notebook.


In [1]:
# NOTE: we deliberately do NOT touch torch/torchvision here.
# Kaggle ships a torch+torchvision+CUDA driver combo that is already matched to the GPU.
# Upgrading torch alone (e.g. with `pip install -U torch`) leaves torchvision on a
# mismatched version, which causes a circular-import crash the moment transformers
# tries to import it:
#   AttributeError: partially initialized module 'torchvision' has no attribute 'extension'
# We don't need torchvision at all for this notebook, so we just leave torch/torchvision alone.
!pip install -q -U datasets requests bitsandbytes "transformers>=4.46.0" "accelerate>=0.34.0" "tokenizers>=0.20.0" sentencepiece evaluate

# IMPORTANT: after this cell finishes, RESTART THE KERNEL/SESSION
# (Kaggle menu -> Run -> "Restart Session", or the restart button in the toolbar),
# then run the notebook from the top again (skipping re-running this pip install cell is fine).
#
# Why: Kaggle base images ship with an older `transformers` that is already imported
# into the running kernel. Without a restart, Python keeps using that stale cached
# module even though pip just installed a newer version on disk, which causes:
#   ModuleNotFoundError: Could not import module 'Qwen2ForCausalLM'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 

## 1. Core Imports and Configuration

In [2]:
import gc
import math
import multiprocessing
import json
import re
import os
import time
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from datasets import load_dataset
import torch

NUM_SAMPLES = 1   # Number of samples to generate per problem
K_VALUE = 1        # K for pass@k metric (pass@1)
MAX_NEW_TOKENS = 512
RESULTS_DIR = "/kaggle/working/data"
os.makedirs(RESULTS_DIR, exist_ok=True)

# HF Login (needed to download the Qwen models / datasets)
# On Kaggle, add your token as a Secret (Add-ons -> Secrets) labeled HF_TOKEN
# and attach it to this notebook.
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    from getpass import getpass
    hf_token = getpass("Enter your Hugging Face token (HF_TOKEN): ")

login(hf_token)


## 2. Models & Datasets to Evaluate

In [3]:
MODELS = [
    {"key": "qwen_base", "hf_id": "Qwen/Qwen2.5-Coder-7B", "is_instruct": False},
    {"key": "qwen_instruct", "hf_id": "Qwen/Qwen2.5-Coder-7B-Instruct", "is_instruct": True},
]

DATASETS = {
    "humaneval": ("openai/openai_humaneval", "test"),
    "humaneval_plus": ("evalplus/humanevalplus", "test"),
}

## 3. Prompt Templates (Zero-shot / Few-shot)

In [4]:
ZERO_SHOT_PROMPT = """You are an expert Python programmer. Write the implementation for the following problem.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Problem:
{prompt}

Code:
"""

FEW_SHOT_PROMPT = """You are an expert Python programmer. You will be provided some examples, and then a problem to solve.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Example 1:
Problem:
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"

Code:
```python
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"
    return a + b
```

Example 2:
Problem:
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"

Code:
```python
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"
    return s == s[::-1]
```

Problem:
{prompt}

Code:
"""

PROMPT_TEMPLATES = {
    "zero_shot": ZERO_SHOT_PROMPT,
    "few_shot": FEW_SHOT_PROMPT,
}

## 4. Helper Functions (execution, pass@k, cleanup)

In [5]:
def run_eval(code, test_code, queue):
    local_scope = {}
    try:
        exec(code, local_scope)
        exec(test_code, local_scope)
        queue.put(True)
    except Exception:
        queue.put(False)

def exec_code(code, test_code, timeout=5):
    """Executes the code and the test, returning True if passes, False otherwise using multiprocessing"""
    queue = multiprocessing.Queue()
    p = multiprocessing.Process(target=run_eval, args=(code, test_code, queue))
    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        p.join()
        return False

    return queue.get() if not queue.empty() else False

def calculate_pass_at_k(n, c, k):
    """
    Calculate pass@k metric.
    n: total samples
    c: correct samples
    k: k in pass@k
    """
    if n - c < k:
        return 1.0

    if c == 0:
        return 0.0
    try:
        return 1.0 - (math.comb(n - c, k) / math.comb(n, k))
    except ValueError:
        return 0.0

def clean_code(generated_text):
    # Regex to extract code between ```python and ```
    match = re.search(r"```python\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)

    # Fallback to general block
    match = re.search(r"```\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)

    return generated_text.strip()

def save_results(results, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

## 5. Generator (handles both base completion-style and instruct chat-style prompting)

In [6]:
class QwenHFGenerator:
    def __init__(self, model, tokenizer, is_instruct=False):
        self.model = model
        self.tokenizer = tokenizer
        self.is_instruct = is_instruct

    def _build_inputs(self, full_prompt):
        if self.is_instruct:
            messages = [{"role": "user", "content": full_prompt}]
            text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            return self.tokenizer(text, return_tensors="pt").to(self.model.device)
        return self.tokenizer(full_prompt, return_tensors="pt").to(self.model.device)

    def generate(self, prompt_template, task_prompt, num_samples=1):
        full_prompt = prompt_template.format(prompt=task_prompt)
        inputs = self._build_inputs(full_prompt)

        samples = []
        for _ in range(num_samples):
            try:
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=0.2,
                    top_p=0.95,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )

                prompt_length = inputs["input_ids"].shape[1]
                generated_ids = outputs[0][prompt_length:]
                generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
                samples.append(generated_text)
            except Exception as e:
                print(f"Error during generation: {e}")
                samples.append("")

        return samples

## 6. Evaluation Loop (loops over models -> datasets -> prompt templates)

In [7]:
def load_model(hf_id):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    print(f"[{hf_id}] Memory footprint: {model.get_memory_footprint() / 1e9:,.1f} GB")
    return model, tokenizer

def unload_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

def evaluate_dataset(generator, dataset, dataset_key, template_name, prompt_template, model_key):
    print(f"\n===== Model [{model_key}] | Dataset [{dataset_key}] | Template [{template_name}] =====")
    results = []
    total_pass_at_k = 0

    for item in tqdm(dataset):
        task_id = item["task_id"]
        prompt = item["prompt"]
        test_cases = item["test"]
        entry_point = item["entry_point"]

        start_time = time.time()
        raw_samples = generator.generate(prompt_template, prompt, num_samples=NUM_SAMPLES)
        time_taken = time.time() - start_time

        correct_count = 0
        cleaned_codes = []

        for raw_text in raw_samples:
            code = clean_code(raw_text)
            eval_code = code + "\n" + test_cases + f"\ncheck({entry_point})\n"

            passed = exec_code(code, eval_code, timeout=5)
            if passed:
                correct_count += 1

            cleaned_codes.append({
                "raw": raw_text,
                "extracted": code,
                "passed": passed
            })

        pass_at_k = calculate_pass_at_k(NUM_SAMPLES, correct_count, K_VALUE)
        total_pass_at_k += pass_at_k

        results.append({
            "task_id": task_id,
            "task_name": entry_point,
            "samples": cleaned_codes,
            "correct_count": correct_count,
            "n": NUM_SAMPLES,
            "pass@k": pass_at_k,
            "time_taken": time_taken
        })

    avg_pass_at_k = total_pass_at_k / len(dataset)
    avg_time = sum(r["time_taken"] for r in results) / len(results)
    print(f"[{model_key}][{dataset_key}][{template_name}] Avg pass@{K_VALUE}: {avg_pass_at_k:.4f} - Avg time: {avg_time:.2f}s/it")

    filename = os.path.join(RESULTS_DIR, f"results_{model_key}_{dataset_key}_{template_name}.json")
    save_results(results, filename)
    print(f"Results saved to {filename}")

    return avg_pass_at_k

In [8]:
# Pre-load all datasets once (small, text-only)
loaded_datasets = {}
for dataset_key, (hf_path, split) in DATASETS.items():
    ds = load_dataset(hf_path, split=split)
    loaded_datasets[dataset_key] = ds
    print(f"[{dataset_key}] {hf_path} -> {len(ds)} problems")

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

[humaneval] openai/openai_humaneval -> 164 problems


README.md:   0%|          | 0.00/551 [00:00<?, ?B/s]

data/test-00000-of-00001-5973903632b82d4(…):   0%|          | 0.00/2.90M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

[humaneval_plus] evalplus/humanevalplus -> 164 problems


In [9]:
summary = {}

for model_cfg in MODELS:
    model_key = model_cfg["key"]
    print(f"\n\n########## Loading model: {model_cfg['hf_id']} ##########")
    model, tokenizer = load_model(model_cfg["hf_id"])
    generator = QwenHFGenerator(model, tokenizer, is_instruct=model_cfg["is_instruct"])

    for dataset_key, dataset in loaded_datasets.items():
        for template_name, prompt_template in PROMPT_TEMPLATES.items():
            avg_pass_at_k = evaluate_dataset(
                generator, dataset, dataset_key, template_name, prompt_template, model_key
            )
            summary[f"{model_key}/{dataset_key}/{template_name}"] = avg_pass_at_k

    unload_model(model)
    print(f"########## Unloaded model: {model_cfg['hf_id']} ##########")



########## Loading model: Qwen/Qwen2.5-Coder-7B ##########


config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

[qwen_instruct][humaneval_plus][zero_shot] Avg pass@1: 0.7927 - Avg time: 9.01s/it
Results saved to /kaggle/working/data/results_qwen_instruct_humaneval_plus_zero_shot.json

===== Model [qwen_instruct] | Dataset [humaneval_plus] | Template [few_shot] =====


 39%|███▉      | 64/164 [09:52<13:27,  8.08s/it]

2
3
0
1
5
2
3
0
1
5


100%|██████████| 164/164 [31:31<00:00, 11.53s/it]


[qwen_instruct][humaneval_plus][few_shot] Avg pass@1: 0.8232 - Avg time: 11.21s/it
Results saved to /kaggle/working/data/results_qwen_instruct_humaneval_plus_few_shot.json
########## Unloaded model: Qwen/Qwen2.5-Coder-7B-Instruct ##########


## 7. Final Summary

In [10]:
print(f"{'Run':45s} pass@{K_VALUE}")
print("-" * 60)
for run_name, score in summary.items():
    print(f"{run_name:45s} {score:.4f}")

save_results(summary, os.path.join(RESULTS_DIR, "summary.json"))
print(f"\nSummary saved to {os.path.join(RESULTS_DIR, 'summary.json')}")

Run                                           pass@1
------------------------------------------------------------
qwen_base/humaneval/zero_shot                 0.7317
qwen_base/humaneval/few_shot                  0.7683
qwen_base/humaneval_plus/zero_shot            0.6890
qwen_base/humaneval_plus/few_shot             0.6951
qwen_instruct/humaneval/zero_shot             0.8415
qwen_instruct/humaneval/few_shot              0.8780
qwen_instruct/humaneval_plus/zero_shot        0.7927
qwen_instruct/humaneval_plus/few_shot         0.8232

Summary saved to /kaggle/working/data/summary.json
